#### Initialize

In [7]:
import sys
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
PARENT = HERE.parent.parent.parent  # server/scripts
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))
LEVEL3_BUCKET_PATH = PARENT / "server/out/places_level3"
LEVEL4_BUCKET_PATH = PARENT / "server/out/places_level4"
LEVEL4_BUCKET_PATH.mkdir(parents=True, exist_ok=True)
LEVEL3_BUCKET = [f for f in LEVEL3_BUCKET_PATH.rglob("*.csv") if f.is_file()]
DF_LEVEL3 = pd.concat([pd.read_csv(f) for f in LEVEL3_BUCKET], ignore_index=True)

#### Wilson Score


Two complementary methods are implemented side-by-side:

| Method | Formula | Strengths | Weaknesses |
|---|---|---|---|
| **Wilson Score** (primary) | Lower bound of 95 % CI for $\hat{p} = (\text{rating}-1)/4$ | Statistically robust; penalises sparse ratings correctly; used by Reddit & IMDB | Slightly complex; favours established places at high confidence |
| **Bayesian Average** (reference) | $\frac{v}{v+m} \cdot R + \frac{m}{v+m} \cdot C$ | Simple; naturally regresses to global mean | Less principled; sensitive to choice of $m$ |

**Wilson Score intuition:** a place with a 5.0 average from 3 reviews is *less trustworthy* than one with 4.8 from 400 reviews. The lower bound of the confidence interval captures exactly that — the more reviews, the tighter the interval and the higher the lower bound.

The `confidence` parameter controls how conservative the ranking is:
- **0.99** (default) — favours well-established places with many reviews  
- **0.95** — gives newer high-rated places a larger boost

In [ ]:
from server.scripts.rank_places_wilson_score.wilson_score import wilson_score
CONFIDENCE = 0.95   # ← adjust: 0.99 = conservative, 0.95 = give newcomers more credit

df_wilson = DF_LEVEL3.copy()
df_wilson = df_wilson.dropna(subset=["rating", "userRatingCount"])#.query("userRatingCount > 0").copy()
df_wilson["capped_ratings"] = df_wilson["userRatingCount"].apply(lambda c: min(c, 1000))  # cap ratings to prevent outliers dominating
df_wilson["wilson_score"] = df_wilson.apply(
    lambda r: wilson_score(r["rating"], int(r["capped_ratings"]), CONFIDENCE), axis=1
)
df_wilson["wilson_quantile"] = df_wilson["wilson_score"].rank(pct=True)
df_wilson.sort_values("wilson_score", ascending=False, inplace=True)
df_wilson.reset_index(drop=True, inplace=True)
df_wilson.index += 1  # 1-based rank index

print(f"Total ranked: {len(df_wilson)}  |  confidence: {CONFIDENCE}")
display(
    df_wilson[
        (df_wilson["wilson_quantile"] >= 0.9) 
    ][["displayName", "primaryTypeDisplayName", "wilson_score", "wilson_quantile"]]
    .head(5)
)

Total ranked: 12320  |  confidence: 0.95


,displayName,primaryTypeDisplayName,wilson_score,wilson_quantile
1,Ethical Bean Company Coffee Shop,Vegan Restaurant,0.996173,0.999838
2,Falafel Zaki Zaki,Vegan Restaurant,0.996173,0.999838
3,Eye Falafel,Falafel Restaurant,0.996173,0.999838
4,Tofu Vegan Charlotte Street,Restaurant,0.996173,0.999838
5,Brazilicious Churros,Restaurant,0.996173,0.999838


#### Export

In [9]:
for seed_id, group in df_wilson.groupby("seed_index"):
    save_path = LEVEL4_BUCKET_PATH / f"{seed_id}.csv"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    group.to_csv(save_path, index=False)